IMPORTS

In [ ]:
from PIL.ImageColor import colormap
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

DEVICE CHECKING

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Working with {DEVICE}...')

IMAGE PROCESSING

In [ ]:
tranform = trnasforms.ToTensor()

train_data = datasets.MNIST(root = './data', train = True, download = True, transfrom = transform)
test_data = datasets.MNIST(root = '/data', train = False, download = True, transform = transform)

BATCH_SIZE = 128

train_loader = DataLoader(train_data, batch_size= BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size= BATCH_SIZE, shuffle=False)

print(f'Number of train samples: {len(train_loader)}')
print(f'Number of test samples: {len(test_loader)}')

VAE CLASS

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim = 2):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 400),
            nn.ReLU(),
        )
        self.mu = nn.Linear(400, latent_dim)
        self.logvar = nn.Linear(400, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 400),
            nn.ReLU(),
            nn.Linear(400, 28 *28),
            nn.Sigmoid(),
        )

    def encode(self, x):
        x = self.encoder(x)
        mu = self.mu(x)
        logvar = self.logvar(x)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, x):
        x = self.decoder(x)
        return x.view(-1, 1, 28, 28)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        z = self.decode(z)
        return z, mu, logvar

In [ ]:
latent_dim = 2
model = VAE(latent_dim).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), 1e-3)

LOSS FUNCTION

In [ ]:
def loss_function(z, x, mu, logvar):
    x = x.view(x.size(0), -1)
    z = z.view(z.size(0), -1)
    reconstruction_loss = F.binary_cross_entropy(z, x, reduction= 'sum')
    KL_divergence_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return reconstruction_loss + KL_divergence_loss, reconstruction_loss, KL_divergence_loss


TRAINER & TESTER FUNCTIONS

In [ ]:
def trainer(epoch):
    model.train()
    total_reconstruction_loss = total_KL_divergence_loss = total_loss = 0

    for x, _ in train_loader:
        x = x.to(DEVICE)
        z, mu, logvar = model(x)
        loss, reconstruction_loss, KL_divergence_loss = loss_function(z, x, mu, logvar)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_reconstruction_loss += reconstruction_loss.item()
        total_KL_divergence_loss += KL_divergence_loss.item()

    n = len(train_data)
    print(f"EPOCH: {epoch}, Total Loss: {total_loss/n:.4f}, Reconstruction Loss: {total_reconstruction_loss/n:.4f}, KL Divergence Loss: {total_KL_divergence_loss/n:.4f}")

def tester():
    model.eval()
    total_reconstruction_loss = total_KL_divergence_loss = total_loss = 0
    with torch.no_grad():
        for x, _ in train_loader:
            x = x.to(DEVICE)
            z, mu, logvar = model(x)
            loss, reconstruction_loss, KL_divergence_loss = loss_function(z, x, mu, logvar)

            total_loss += loss.item()
            total_reconstruction_loss += reconstruction_loss.item()
            total_KL_divergence_loss += KL_divergence_loss.item()

    n = len(train_data)
    print(f"\nTesting Results:\nTotal Loss: {total_loss/n:.4f}, Reconstruction Loss: {total_reconstruction_loss/n:.4f}, KL Divergence Loss: {total_KL_divergence_loss/n:.4f}")

TRAINING THE MODEL

In [ ]:
epochs = 7
for i in range(1, epochs + 1):
    trainer(epochs)
    tester()

SHOWING RECONSTRUCTION

In [ ]:
def show_reconstructions():
    
    model.eval()
    x, _ = next(iter(test_loader))
    x = x.to(DEVICE)

    with torch.no_grad():
        y, _, _ = model(x)

    y = y.cpu().numpy()
    x = x.cpu().numpy()

    samples = 8

    plt.figure(figsize = (10, 5))
    for i in range(samples):
        plt.subplot(2, samples, i+1)
        plt.imshow(x[i][0], cmap = 'gray')
        plt.axis('off')

        plt.subplot(2, samples, n+i+1)
        plt.imshow(y[i][0], cmap = 'gray')
        plt.axis('off')
    
    plt.suptitle("Original (top) vs Reconstruction (bottom)")
    plt.show()

show_reconstructions()

PLOTTING IN LATENT SPACE

In [ ]:
def plot_in_latent_space(num_batches = 30):
    model.eval()
    zs, labels = [], []

    with torch.no_grad():
        for i, (x, y) in enumerate(test_loader):
            if i>num_batches:
                break
        x = x.to(DEVICE)
        mu, _ = model.encode(x)
        zs.append(mu.cpu().numpy())
        labels.append(y.numpy())
    
    zs = np.concatenate(zs, axis = 0)
    labels = np.concatenate(labels, axis = 0)

    plt.figure(figsize=(7, 7))
    scatter = plt.scatter(zs[:, 0], zs[:, 1], c = labels, cmap ="tab10", s = 10)
    plt.colorbar(scatter)
    plt.xlabel("z1")
    plt.ylabel("z2")
    plt.grid(True)
    plt.title("2D Latent Space")
    plt.show()

plot_in_latent_space()

SAMPLING FROM LATENT SPACE

In [ ]:
def sample_latent_grid():
    model.eval()
    n = 5
    x = np.linspace(-3, 3, n)
    y = np.linspace(-3, 3, n)
    plt.figure(figsize=(6, 6))
    with torch.no_grad():
        for i, p1 in enumerate(x):
            for j, p2 in enumerate(y):
                z = torch.tensor([[x,y]], dtype = torch.float32).to(DEVICE)
                result = model.decode(z)[0, 0].cpu().numpy()
                plt.subplot(n, n, i * n + j + 1)
                plt.imshow(result, cmap = "gray")
                plt.axis("off")
    
    plt.suptitle("Sampling from the 2D Latent Space")
    plt.show()

sample_latent_grid()

FUNCTION FOR COLLECTING SAMPLES FOR THE FOLLOWING plot_hist() FUNCTION

In [ ]:
from math import sqrt, pi, exp

def collect_latent_samples(num_batches=50, use_mu=True):
    
    model.eval()
    zs = []

    with torch.no_grad():
        for i, (x, _) in enumerate(test_loader):
            if i >= num_batches:
                break
            x = x.to(device)
            mu, logvar = model.encode(x)
            if use_mu:
                z = mu
            else:
                # sample z = mu + sigma * eps
                std = torch.exp(0.5 * logvar)
                eps = torch.randn_like(std)
                z = mu + std * eps
            zs.append(z.cpu().numpy())

    if len(zs) == 0:
        return None

    zs = np.concatenate(zs, axis=0)   # shape [N, latent_dim]
    return zs


COMPARING WITH GAUSSIAN DISTRIBUTION

In [ ]:
def normal_pdf(x):
    # N(0,1)
    return (1.0 / sqrt(2 * pi)) * np.exp(-0.5 * x**2)

def plot_hist(z_samples, bins = 40):
    
    if z_samples is None:
        print("No latent samples to plot.")
        return

    z1 = z_samples[:, 0]
    z2 = z_samples[:, 1]

    mean_z1, std_z1 = np.mean(z1), np.std(z1)
    mean_z2, std_z2 = np.mean(z2), np.std(z2)

    xs = np.linspace(-4, 4, 400)
    pdf = normal_pdf(xs)

    plt.figure(figsize=(12, 4))

    # z1 histogram
    plt.subplot(1, 3, 1)
    plt.hist(z1, bins = bins, density = True, alpha = 0.6)
    plt.plot(xs, pdf, linewidth = 2)
    plt.title("z1 distribution")
    plt.xlabel("z1")
    plt.ylabel("density")

    # z2 histogram
    plt.subplot(1, 3, 2)
    plt.hist(z2, bins = bins, density = True, alpha = 0.6)
    plt.plot(xs, pdf, linewidth = 2)
    plt.title("z2 distribution")
    plt.xlabel("z2")
    plt.ylabel("density")

    # 2D scatter
    plt.subplot(1, 3, 3)
    plt.scatter(z1, z2, s = 4, alpha = 0.5)
    plt.title("2D latent scatter")
    plt.xlabel("z1")
    plt.ylabel("z2")
    plt.xlim(-4, 4)
    plt.ylim(-4, 4)

    plt.tight_layout()
    plt.show()

plot_hist(collect_latent_samples())